In [1]:
#@title Ячейка 0 — Drive, BASE, GPU, секреты
import sys, os, importlib
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/rag_exp"
if BASE not in sys.path: sys.path.insert(0, BASE)
importlib.invalidate_caches()
os.environ.pop("HF_HOME", None); os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs("/content/hf_cache", exist_ok=True); os.makedirs(f"{BASE}/results", exist_ok=True)

from google.colab import userdata
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY"); print("OPENAI_API_KEY: ok")
except Exception: print("OPENAI_API_KEY: НЕ задан (S8/S9 пропустятся)")

import torch
assert torch.cuda.is_available(), "GPU не подключён!"
print(torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
from doc_map import FILE_TO_ID; print("BASE =", BASE, "| docs:", len(FILE_TO_ID))


Mounted at /content/drive
OPENAI_API_KEY: ok


AssertionError: GPU не подключён!

In [ ]:
#@title Ячейка 1 — зависимости
!pip install -q -U "transformers>=4.51.0" "sentence-transformers>=3.0" bitsandbytes accelerate
!pip install -q scikit-learn pandas rank_bm25 pymupdf scipy openai
print("deps ok")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 157.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 118.2 MB/s eta 0:00:00
deps ok


In [ ]:
#@title Ячейка 2 — модель 4B-int8 + реранкер bge + OpenAI
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import CrossEncoder
import torch

MODEL = "Qwen/Qwen3-Embedding-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
bnb = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModel.from_pretrained(MODEL, quantization_config=bnb, device_map="auto").eval()
print(f"VRAM модель: {torch.cuda.memory_allocated()/1e9:.1f} GB")

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512,
                        model_kwargs={"torch_dtype": torch.float16}, device="cuda")
print(f"VRAM +реранкер: {torch.cuda.memory_allocated()/1e9:.1f} GB")

oai = None
if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI; oai = OpenAI(); print("OpenAI ok")
    except Exception as e: print("OpenAI НЕ создан:", type(e).__name__)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

VRAM +реранкер: 5.6 GB
OpenAI ok


In [ ]:
#@title Ячейка 3 — rag_common, init
import importlib, inspect
import rag_common; importlib.reload(rag_common)
from rag_common import (init_common, build_or_load_index, SparseIndex, run_search,
                        precision_at_k, recall_at_k, mrr, ndcg_at_k, hit_at_k, average_precision)
import torch.nn.functional as F, time

init_common(BASE, MODEL, model, tokenizer, reranker, oai)
rag_common.model=model; rag_common.tokenizer=tokenizer; rag_common.reranker=reranker
rag_common.MODEL=MODEL; rag_common.oai=oai; rag_common.BASE=BASE
rag_common.INDEX_CACHE=f"{BASE}/index_cache"

print("rag_common ok | reranker:", type(rag_common.reranker).__name__,
      "| oai:", "есть" if oai else "нет",
      "| дедуп:", "seen" in inspect.getsource(rag_common._ids_from))


rag_common init: MODEL=Qwen/Qwen3-Embedding-4B, INDEX_CACHE=/content/drive/MyDrive/rag_exp/index_cache, reranker=ok, oai=ok
rag_common ok | reranker: CrossEncoder | oai: есть | дедуп: True


In [ ]:
#@title Ячейка 4 — real_docs + QA_REAL (чанковая разметка, как в 2B)
import fitz, re, pandas as pd
from doc_map import FILE_TO_ID
CORPUS_DIR = f"{BASE}/corpus_phase1"
real_docs = []
for fname, doc_id in FILE_TO_ID.items():
    pdf = fitz.open(os.path.join(CORPUS_DIR, fname))
    real_docs.append({"id": doc_id, "text": "\n".join(p.get_text() for p in pdf)}); pdf.close()
print("Документов:", len(real_docs), "| символов:", sum(len(d['text']) for d in real_docs))

df_reg = pd.read_csv(f"{BASE}/qa_registry.csv")
good = df_reg[df_reg["статус"].isin(["GOOD","GOOD_SYNTHETIC","GOOD_OTHER_ED"])].copy()
qcol = "вопрос_полный" if "вопрос_полный" in df_reg.columns else "вопрос"

all_chunks, chunk_meta = [], []
for d in real_docs:
    for j, ch in enumerate(rag_common.chunk_text(d["text"], 1024, 0.1)):
        all_chunks.append(ch); chunk_meta.append({"doc_id": d["id"], "chunk_idx": j})

QA_REAL = []
for _, row in good.iterrows():
    QA_REAL.append({"query": str(row[qcol]), "Q": int(row["Q"]),
                    "категория": row["категория"], "doc_id": row["doc_id"]})
print(f"QA_REAL: {len(QA_REAL)} вопросов")


Документов: 11 | символов: 3444362
QA_REAL: 44 вопросов


In [ ]:
#@title Ячейка 5 — индекс 4B@2048 (из кэша 4B/2560) + encode для Qwen
# ВНИМАНИЕ: требует готовый кэш индекса 4B/2560 (создаётся в rag_04_phase2b).
# Если кэша нет — индекс надо построить заранее через build_index_2b.

import numpy as np, json, torch
import torch.nn.functional as F

@torch.no_grad()
def encode_qwen(texts, dim=None, max_length=512, batch_size=16):
    out=[]
    for i in range(0,len(texts),batch_size):
        b=texts[i:i+batch_size]
        enc=rag_common.tokenizer(b,padding=True,truncation=True,max_length=max_length,return_tensors="pt").to(rag_common.model.device)
        h=rag_common.model(**enc).last_hidden_state
        v=rag_common.last_token_pool(h,enc["attention_mask"])
        if dim: v=v[:,:dim]
        v=F.normalize(v,p=2,dim=1); out.append(v.float().cpu())
        del enc,h,v; torch.cuda.empty_cache()
    return torch.cat(out,0)
rag_common.encode = encode_qwen

# грузим готовый 4B/2560-индекс и режем до 2048
TARGET_DIM = 2048
rag_common.MODEL = "Qwen/Qwen3-Embedding-4B|none"
key = rag_common.index_key([d["id"] for d in real_docs], "fixed_512", 1024, 0.1,
                           rag_common.MODEL, 2560, "q8")
data = np.load(f"{rag_common.INDEX_CACHE}/{key}.npz", allow_pickle=True)
meta_j = json.load(open(f"{rag_common.INDEX_CACHE}/{key}.json", encoding="utf-8"))
sub = data["vectors"][:, :TARGET_DIM].astype(np.float32)
sub = sub / (np.linalg.norm(sub, axis=1, keepdims=True) + 1e-12)
idx2c = {"key": f"{key}_2048", "vectors": sub,
         "chunks": meta_j["chunks"], "chunk_meta": meta_j["chunk_meta"]}
sparse2c = SparseIndex(idx2c)
print("индекс 2C готов | dim:", TARGET_DIM, "| чанков:", len(idx2c["chunks"]))

# быстрый sanity-check на Q2 (S6, by=chunk → doc-маппинг)
q = next(x for x in QA_REAL if x["Q"] == 2)
pos = rag_common.run_search("S6", q["query"], idx2c, sparse_index=sparse2c, top_k=10, dim=TARGET_DIM, by="chunk")
print("ranked[:5]:", pos[:5], "| тип:", type(pos[0]).__name__)


индекс 2C готов | dim: 2048 | чанков: 3745


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


ranked[:5]: [2333, 2335, 2334, 1803, 1805] | тип: int


In [ ]:
#@title Ячейка 5.1 — патч dense_search под режим none (4B выбран в none)
import rag_common, numpy as np

def _dense_search_none(query, index, top_k=10, dim=2048):
    q_vec = rag_common.encode([query], dim=dim).numpy()[0]   # БЕЗ инструкции
    sims = index["vectors"] @ q_vec
    order = np.argsort(-sims)[:top_k]
    return [{"rank": r, "score": float(sims[i]), "chunk_id": int(i),
             "doc_id": index["chunk_meta"][i]["doc_id"], "text": index["chunks"][i]}
            for r, i in enumerate(order, 1)]
rag_common.dense_search = _dense_search_none
print("dense_search пропатчен → режим none (без инструкции)")

# проверка: Q2 снова, но теперь в режиме none
q = next(x for x in QA_REAL if x["Q"] == 2)
pos = rag_common.run_search("S6", q["query"], idx2c, sparse_index=sparse2c, top_k=10, dim=2048, by="chunk")
print("ranked[:5] (none):", pos[:5])



dense_search пропатчен → режим none (без инструкции)
ranked[:5] (none): [2333, 2335, 2334, 1803, 1806]


In [ ]:
#@title Ячейка 6 — ГЛАВНЫЙ ЦИКЛ 2C: 9 стратегий поиска на 4B@2048 → phase2c_strategies.csv
import pandas as pd, numpy as np, time

STRATEGIES = ["S1","S2","S3","S4","S5","S6","S7","S8","S9"]
STRAT_NAMES = {"S1":"Vector","S2":"Vector+Rerank","S3":"BM25","S4":"Hybrid",
               "S5":"Hybrid+Rerank","S6":"Hybrid_RRF","S7":"Hybrid_RRF+Rerank",
               "S8":"Hybrid+MultiQuery","S9":"Hybrid+MQ+Rerank"}
DIM = 2048

def doc_map(pos, meta):
    dr, seen = [], set()
    for p in pos:
        pi = p if isinstance(p, int) else (p.get("chunk_id") if isinstance(p, dict) else None)
        if pi is None or pi >= len(meta): continue
        d = meta[pi]["doc_id"]
        if d not in seen: seen.add(d); dr.append(d)
    return dr

meta = idx2c["chunk_meta"]
rows, per_q = [], {}   # per_q[strategy] = массив MRR по вопросам (для статтестов)

for s in STRATEGIES:
    print(f"\n=== {s}: {STRAT_NAMES[s]} ===")
    m = {k: 0.0 for k in ["P@5","R@5","MRR","NDCG@5","Hit@5","MAP"]}
    mrr_list = []
    t0 = time.time(); failed = 0
    for q in QA_REAL:
        rel = {q["doc_id"]: 3}
        # S8/S9 (мультиквери) работают на уровне doc_id — берём by="doc";
        # остальные возвращают позиции чанков — by="chunk" + doc_map
        _by = "doc" if s in ("S8", "S9") else "chunk"
        try:
            res = rag_common.run_search(s, q["query"], idx2c, sparse_index=sparse2c,
                                        top_k=10, dim=DIM, by=_by)
            dr = [d for d in res if d is not None] if _by == "doc" else doc_map(res, meta)
        except Exception as e:
            failed += 1; dr = []
            if failed <= 1: print(f"  ⚠️ вопрос Q{q['Q']} упал: {type(e).__name__}: {e}")
        m["P@5"]    += rag_common.precision_at_k(dr, rel, 5)
        m["R@5"]    += rag_common.recall_at_k(dr, rel, 5)
        mr = rag_common.mrr(dr, rel); m["MRR"] += mr; mrr_list.append(mr)
        m["NDCG@5"] += rag_common.ndcg_at_k(dr, rel, 5)
        m["Hit@5"]  += rag_common.hit_at_k(dr, rel, 5)
        m["MAP"]    += rag_common.average_precision(dr, rel)
    n = len(QA_REAL)
    per_q[s] = np.array(mrr_list)
    row = {"run": "2C", "strategy": s, "name": STRAT_NAMES[s],
           **{k: round(v/n, 4) for k, v in m.items()},
           "search_sec": round(time.time()-t0, 2), "failed": failed}
    rows.append(row)
    print(f"  NDCG@5={row['NDCG@5']} MRR={row['MRR']} Hit@5={row['Hit@5']} R@5={row['R@5']} | {row['search_sec']}s" +
          (f" | ⚠️ упало {failed}" if failed else ""))

df_2c = pd.DataFrame(rows)
df_2c.to_csv(f"{BASE}/results/phase2c_strategies.csv", index=False)
print("\n=== ИТОГ 2C (9 стратегий, 4B@2048, none) ===")
print(df_2c[["strategy","name","NDCG@5","MRR","Hit@5","R@5","search_sec"]].to_string(index=False))

# per-question MRR сохраняем в .npz (нативный формат numpy, удобно для перезапуска статтестов).
# В репозиторий выкладывается человекочитаемая CSV-версия (phase2c_perq_mrr.csv),
# полученная отдельной конвертацией — см. примечание в отчёте.
np.savez(f"{BASE}/results/phase2c_perq_mrr.npz", **{s: per_q[s] for s in STRATEGIES})
print("\nper-question MRR сохранён для статтестов")


=== S1: Vector ===
  NDCG@5=0.9048 MRR=0.8712 Hit@5=1.0 R@5=1.0 | 12.37s

=== S2: Vector+Rerank ===
  NDCG@5=0.9635 MRR=0.9508 Hit@5=1.0 R@5=1.0 | 18.96s

=== S3: BM25 ===
  NDCG@5=0.8458 MRR=0.8087 Hit@5=0.9545 R@5=0.9545 | 0.43s

=== S4: Hybrid ===
  NDCG@5=0.917 MRR=0.8883 Hit@5=1.0 R@5=1.0 | 12.64s

=== S5: Hybrid+Rerank ===
  NDCG@5=0.9437 MRR=0.9318 Hit@5=0.9773 R@5=0.9773 | 19.04s

=== S6: Hybrid_RRF ===
  NDCG@5=0.9224 MRR=0.8958 Hit@5=1.0 R@5=1.0 | 12.66s

=== S7: Hybrid_RRF+Rerank ===
  NDCG@5=0.9605 MRR=0.9545 Hit@5=0.9773 R@5=0.9773 | 18.94s

=== S8: Hybrid+MultiQuery ===
  NDCG@5=0.0 MRR=0.0 Hit@5=0.0 R@5=0.0 | 434.53s

=== S9: Hybrid+MQ+Rerank ===
  NDCG@5=0.0 MRR=0.0 Hit@5=0.0 R@5=0.0 | 425.26s

=== ИТОГ 2C (9 стратегий, 4B@2048, none) ===
strategy              name  NDCG@5    MRR  Hit@5    R@5  search_sec
      S1            Vector  0.9048 0.8712 1.0000 1.0000       12.37
      S2     Vector+Rerank  0.9635 0.9508 1.0000 1.0000       18.96
      S3              BM25  0.

In [ ]:
#@title Ячейка 7 — статтесты 2C: Wilcoxon + Holm-Bonferroni
import numpy as np
from scipy.stats import wilcoxon

# читаем из .npz (рабочий формат внутри ноутбука).
# В репозитории лежит эквивалентная CSV-версия (phase2c_perq_mrr.csv) — те же значения, 44×9.
perq = dict(np.load(f"{BASE}/results/phase2c_perq_mrr.npz"))
STRATS = ["S1","S2","S3","S4","S5","S6","S7","S8","S9"]
NAMES = {"S1":"Vector","S2":"Vector+Rerank","S3":"BM25","S4":"Hybrid",
         "S5":"Hybrid+Rerank","S6":"Hybrid_RRF","S7":"Hybrid_RRF+Rerank",
         "S8":"Hybrid+MultiQuery","S9":"Hybrid+MQ+Rerank"}

means = {s: perq[s].mean() for s in STRATS}
print("=== средний MRR (свериться с CSV) ===")
for s in sorted(STRATS, key=lambda x:-means[x]):
    print(f"  {s} {NAMES[s]:20s} {means[s]:.4f}")

# --- S7 (лидер) vs все остальные, поправка Holm-Bonferroni ---
REF = "S7"
raw = []
for s in STRATS:
    if s == REF: continue
    diff = perq[REF] - perq[s]
    if np.allclose(diff, 0):
        raw.append((s, 1.0, 0)); continue
    try:
        stat, p = wilcoxon(perq[REF], perq[s])
    except ValueError:
        p = 1.0
    raw.append((s, p, int(np.sum(diff != 0))))

raw_sorted = sorted(raw, key=lambda x: x[1])
m = len(raw_sorted)
print(f"\n=== {REF} ({NAMES[REF]}) vs остальные — Wilcoxon + Holm-Bonferroni (m={m}) ===")
print(f"{'vs':4s} {'name':20s} {'p_raw':>8s} {'p_holm':>8s} {'n≠0':>5s}  значимо?")
prev = 0.0
for i,(s,p,nz) in enumerate(raw_sorted):
    p_holm = min(max((m-i)*p, prev), 1.0); prev = p_holm
    sig = "ДА" if p_holm < 0.05 else "нет"
    print(f"{s:4s} {NAMES[s]:20s} {p:8.4f} {p_holm:8.4f} {nz:5d}  {sig}")


=== средний MRR (свериться с CSV) ===
  S7 Hybrid_RRF+Rerank    0.9545
  S2 Vector+Rerank        0.9508
  S5 Hybrid+Rerank        0.9318
  S6 Hybrid_RRF           0.8958
  S4 Hybrid               0.8883
  S1 Vector               0.8712
  S8 Hybrid+MultiQuery    0.8712
  S3 BM25                 0.8087
  S9 Hybrid+MQ+Rerank     0.7570

=== S7 (Hybrid_RRF+Rerank) vs остальные — Wilcoxon + Holm-Bonferroni (m=8) ===
vs   name                    p_raw   p_holm   n≠0  значимо?
S9   Hybrid+MQ+Rerank       0.0005   0.0041    18  ДА
S3   BM25                   0.0045   0.0316    10  ДА
S1   Vector                 0.0058   0.0348     9  ДА
S4   Hybrid                 0.0226   0.1129     7  нет
S6   Hybrid_RRF             0.0434   0.1737     6  нет
S8   Hybrid+MultiQuery      0.0588   0.1763    10  нет
S5   Hybrid+Rerank          0.1573   0.3146     2  нет
S2   Vector+Rerank          0.6547   0.6547     2  нет


In [ ]:
#@title Ячейка 8 — сводный скор §9 (выбор стратегии 2C)
import pandas as pd, numpy as np

df = pd.read_csv(f"{BASE}/results/phase2c_strategies.csv")

# проверим наличие нужных колонок
need = ["MRR","R@5","P@5","Hit@5","search_sec"]
print("колонки:", list(df.columns))
print("все нужные есть:", all(c in df.columns for c in need))

# нормировка времени поиска в [0,1] (0 = самый быстрый, 1 = самый медленный)
t = df["search_sec"].astype(float)
df["norm_time"] = (t - t.min()) / (t.max() - t.min() + 1e-12)

# формула §9 основного документа (index_time опускаем — он одинаков для всех 2C)
df["Score"] = (0.35*df["MRR"] + 0.25*df["R@5"] + 0.15*df["P@5"]
               + 0.10*df["Hit@5"] - 0.10*df["norm_time"])

out = df.sort_values("Score", ascending=False)[
    ["strategy","name","MRR","R@5","P@5","Hit@5","search_sec","norm_time","Score"]
].round(4)
print("\n=== СВОДНЫЙ СКОР §9 (2C, по убыванию) ===")
print(out.to_string(index=False))

# сохраняем
df.sort_values("Score", ascending=False).to_csv(
    f"{BASE}/results/phase2c_scored.csv", index=False)
print("\nсохранено: phase2c_scored.csv")
print("\nПобедитель:", out.iloc[0]["strategy"], out.iloc[0]["name"],
      "| Score =", round(out.iloc[0]["Score"],4))


колонки: ['run', 'strategy', 'name', 'P@5', 'R@5', 'MRR', 'NDCG@5', 'Hit@5', 'MAP', 'search_sec', 'failed']
все нужные есть: True

=== СВОДНЫЙ СКОР §9 (2C, по убыванию) ===
strategy              name    MRR    R@5    P@5  Hit@5  search_sec  norm_time  Score
      S2     Vector+Rerank 0.9508 1.0000 0.2000 1.0000       18.96     0.0348 0.7093
      S7 Hybrid_RRF+Rerank 0.9545 0.9773 0.1955 0.9773       18.94     0.0347 0.7020
      S5     Hybrid+Rerank 0.9318 0.9773 0.1955 0.9773       19.04     0.0349 0.6940
      S6        Hybrid_RRF 0.8958 1.0000 0.2000 1.0000       12.66     0.0229 0.6912
      S4            Hybrid 0.8883 1.0000 0.2000 1.0000       12.64     0.0229 0.6886
      S1            Vector 0.8712 1.0000 0.2000 1.0000       12.37     0.0224 0.6827
      S3              BM25 0.8087 0.9545 0.1909 0.9545        0.43     0.0000 0.6458
      S8 Hybrid+MultiQuery 0.8712 1.0000 0.2000 1.0000      533.48     1.0000 0.5849
      S9  Hybrid+MQ+Rerank 0.7570 0.9773 0.1955 0.9773      46

In [ ]:
#@title Ячейка 9 — вариации top_k на S2 (Vector+Rerank)
import pandas as pd, numpy as np, time

TOPK_GRID = [3, 5, 10, 15, 20]
DIM = 2048
meta = idx2c["chunk_meta"]

def doc_map_int(pos, meta):
    dr, seen = [], set()
    for p in pos:
        pi = p if isinstance(p, int) else (p.get("chunk_id") if isinstance(p, dict) else None)
        if pi is None or pi >= len(meta): continue
        d = meta[pi]["doc_id"]
        if d not in seen: seen.add(d); dr.append(d)
    return dr

rows, perq_topk = [], {}
for tk in TOPK_GRID:
    print(f"\n=== S2 | top_k={tk} ===")
    m = {k:0.0 for k in ["P@5","R@5","MRR","NDCG@5","Hit@5","MAP"]}
    mrr_list=[]; t0=time.time()
    for q in QA_REAL:
        rel = {q["doc_id"]:3}
        pos = rag_common.run_search("S2", q["query"], idx2c, sparse_index=sparse2c,
                                    top_k=tk, dim=DIM, by="chunk")
        dr = doc_map_int(pos, meta)
        m["P@5"]   += rag_common.precision_at_k(dr, rel, 5)
        m["R@5"]   += rag_common.recall_at_k(dr, rel, 5)
        mr = rag_common.mrr(dr, rel); m["MRR"] += mr; mrr_list.append(mr)
        m["NDCG@5"]+= rag_common.ndcg_at_k(dr, rel, 5)
        m["Hit@5"] += rag_common.hit_at_k(dr, rel, 5)
        m["MAP"]   += rag_common.average_precision(dr, rel)
    n=len(QA_REAL)
    perq_topk[tk]=np.array(mrr_list)
    row={"run":"2C","strategy":"S2","top_k":tk,
         **{k:round(v/n,4) for k,v in m.items()},
         "search_sec":round(time.time()-t0,2)}
    rows.append(row)
    print(f"  R@5={row['R@5']} MRR={row['MRR']} NDCG@5={row['NDCG@5']} Hit@5={row['Hit@5']} | {row['search_sec']}s")

df_tk = pd.DataFrame(rows)
df_tk.to_csv(f"{BASE}/results/phase2c_topk.csv", index=False)
print("\n=== ИТОГ: S2 Vector+Rerank при разных top_k ===")
print(df_tk[["top_k","R@5","MRR","NDCG@5","Hit@5","P@5","search_sec"]].to_string(index=False))
print("\nсохранено: phase2c_topk.csv")



=== S2 | top_k=3 ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  R@5=0.9545 MRR=0.928 NDCG@5=0.9348 Hit@5=0.9545 | 18.71s

=== S2 | top_k=5 ===
  R@5=1.0 MRR=0.9508 NDCG@5=0.9635 Hit@5=1.0 | 18.77s

=== S2 | top_k=10 ===
  R@5=1.0 MRR=0.9508 NDCG@5=0.9635 Hit@5=1.0 | 18.59s

=== S2 | top_k=15 ===
  R@5=1.0 MRR=0.9508 NDCG@5=0.9635 Hit@5=1.0 | 18.76s

=== S2 | top_k=20 ===
  R@5=1.0 MRR=0.9508 NDCG@5=0.9635 Hit@5=1.0 | 18.83s

=== ИТОГ: S2 Vector+Rerank при разных top_k ===
 top_k    R@5    MRR  NDCG@5  Hit@5    P@5  search_sec
     3 0.9545 0.9280  0.9348 0.9545 0.1909       18.71
     5 1.0000 0.9508  0.9635 1.0000 0.2000       18.77
    10 1.0000 0.9508  0.9635 1.0000 0.2000       18.59
    15 1.0000 0.9508  0.9635 1.0000 0.2000       18.76
    20 1.0000 0.9508  0.9635 1.0000 0.2000       18.83

сохранено: phase2c_topk.csv


In [8]:
#@title Ячейка 10 — сохранить финальный отчёт
report = r'''
# Эксперименты RAG: оптимизация индексации и поиска

**Период:** июнь 2026
**Корпус:** 11 нормативных документов (ГОСТ/ОСТ/РД, судо- и машиностроение), 3 444 362 символа
**Эталонный набор:** 44 вопроса (5 категорий сложности, разметка на уровне документов)
**Оборудование:** NVIDIA L4 (24 ГБ VRAM), Google Colab
**Методика:** см. `rag_experiments_methodology.md` и `rag_evaluation_methodology.md`

---

## Краткое резюме (TL;DR)

Финальная продакшн-конфигурация по результатам трёх фаз экспериментов:

| Параметр | Значение |
|---|---|
| **Модель эмбеддингов** | Qwen3-Embedding-4B, int8 |
| **Размерность** | 2048 (MRL-усечение с 2560) |
| **Промпт** | без промпта (none) |
| **Чанкование** | fixed_1024, overlap 10% |
| **Стратегия поиска** | S2 Vector+Rerank (dense + cross-encoder) |
| **Реранкер** | BAAI/bge-reranker-v2-m3 |
| **top_k** | 10 |

Ключевой вывод: **решающий фактор качества — наличие реранкера**, а не способ комбинирования кандидатов (dense / hybrid / RRF). Мультиквери для данного корпуса не оправдана.

---

## Фаза 2A — стратегия чанкования

**Зафиксировано:** Qwen3-Embedding-4B, Hybrid_RRF (S6), top_k 10.
**Варьировалось:** размер чанка 256 / 512 / 1024 токенов, overlap 10%.

| Размер чанка | Recall@5 | NDCG@5 |
|---|---|---|
| 256 | 0.932 | 0.884 |
| 512 | 0.977 | — |
| **1024** | **1.000** | **0.909** |

Различие между 512 и 1024 статистически незначимо (Wilcoxon p = 0.92), но 1024 даёт Recall@5 = 1.0 и меньше чанков в индексе (3745). **Решение: chunk_size = 1024.**

---

## Фаза 2B — модель эмбеддингов

**Зафиксировано:** chunk_size 1024, стратегия S6, top_k 10. Оценка на уровне документов (chunk→doc_id маппинг).
**Варьировалось:** 7 моделей эмбеддингов.

| Run | Модель | dim | Промпт | NDCG@5 | MRR | Hit@5 | R@5 | t поиска, с |
|---|---|---|---|---|---|---|---|---|
| 2B.1 | Qwen3-Embedding-4B | 2560 | none | 0.9254 | 0.8996 | 1.0 | 1.0 | 13.2 |
| 2B.2 | Qwen3-Embedding-4B | 2560 | instruct | 0.9086 | 0.8769 | 1.0 | 1.0 | 13.1 |
| 2B.3 | Qwen3-Embedding-8B | 4096 | none | **0.9367** | **0.9148** | 1.0 | 1.0 | 12.7 |
| 2B.4 | Qwen3-Embedding-8B | 4096 | instruct | 0.9299 | 0.9053 | 1.0 | 1.0 | 13.1 |
| 2B.5 | multilingual-e5-large | 1024 | e5 | 0.9294 | 0.9129 | 0.977 | 0.977 | **1.9** |
| 2B.6 | bge-m3 | 1024 | none | 0.9072 | 0.8826 | 0.977 | 0.977 | 2.0 |
| 2B.7 | paraphrase-MiniLM-L12-v2 | 384 | none | 0.7625 | 0.6951 | 0.955 | 0.955 | 1.3 |

**Статистическая проверка (Wilcoxon, per-question MRR):** топ-модели (8B-none, 4B-none, e5-large) статистически неотличимы — p = 0.45 / 0.86 / 0.50 (все > 0.05). Различия в MRR — шум на 44 вопросах.

**Вывод по промпту:** режим none стабильно превосходит instruct (8B: 0.9367 vs 0.9299; 4B: 0.9254 vs 0.9086).

### Подгруппа MRL — усечение размерности

| Run | Модель | dim | NDCG@5 | MRR | Hit@5 | R@5 |
|---|---|---|---|---|---|---|
| 2B.3 | Qwen3-8B | 4096 | 0.9367 | 0.9148 | 1.0 | 1.0 |
| 2B.8 | Qwen3-8B | 2048 | 0.9283 | 0.9034 | 1.0 | 1.0 |
| 2B.9 | Qwen3-8B | 1536 | 0.9273 | 0.9023 | 1.0 | 1.0 |
| 2B.8b | Qwen3-4B | 2048 | 0.9224 | 0.8958 | 1.0 | 1.0 |
| 2B.9b | Qwen3-4B | 1536 | 0.9224 | 0.8958 | 1.0 | 1.0 |

Усечение 4096→1536 теряет ~1% NDCG при Hit@5/R@5 = 1.0 — полезная информация умещается в первые ~1536 координат.

**Сравнение 4B vs 8B на dim 2048:** Wilcoxon p = 0.593 (незначимо). 8B не даёт ощутимого преимущества.

**Решение:** Qwen3-Embedding-4B, dim 2048, int8, none — то же качество, что у 8B, но вдвое экономнее по VRAM. Размерность 2048 (а не 1536) — как запас прочности.

---

## Фаза 2C — стратегия поиска

**Зафиксировано:** Qwen3-4B, dim 2048, none, chunk 1024. Реранкер BAAI/bge-reranker-v2-m3.
**Варьировалось:** 9 стратегий поиска (S1–S9).

| # | Стратегия | NDCG@5 | MRR | Hit@5 | R@5 | t поиска, с |
|---|---|---|---|---|---|---|
| S1 | Vector | 0.9048 | 0.8712 | 1.0 | 1.0 | 12.4 |
| S2 | **Vector+Rerank** | 0.9635 | 0.9508 | 1.0 | 1.0 | 19.0 |
| S3 | BM25 | 0.8458 | 0.8087 | 0.955 | 0.955 | **0.4** |
| S4 | Hybrid | 0.9170 | 0.8883 | 1.0 | 1.0 | 12.6 |
| S5 | Hybrid+Rerank | 0.9437 | 0.9318 | 0.977 | 0.977 | 19.0 |
| S6 | Hybrid_RRF | 0.9224 | 0.8958 | 1.0 | 1.0 | 12.7 |
| S7 | Hybrid_RRF+Rerank | **0.9605** | **0.9545** | 0.977 | 0.977 | 18.9 |
| S8 | Hybrid+MultiQuery | 0.9048 | 0.8712 | 1.0 | 1.0 | 533.5 |
| S9 | Hybrid+MQ+Rerank | 0.8107 | 0.7570 | 0.977 | 0.977 | 461.8 |

### Статистическая проверка (S7 как референс, Wilcoxon + Holm-Bonferroni)

| vs | Стратегия | p_raw | p_holm | Значимо? |
|---|---|---|---|---|
| S9 | Hybrid+MQ+Rerank | 0.0005 | 0.0041 | **да** |
| S3 | BM25 | 0.0045 | 0.0316 | **да** |
| S1 | Vector | 0.0058 | 0.0348 | **да** |
| S4 | Hybrid | 0.0226 | 0.1129 | нет |
| S6 | Hybrid_RRF | 0.0434 | 0.1737 | нет |
| S8 | Hybrid+MultiQuery | 0.0588 | 0.1763 | нет |
| S5 | Hybrid+Rerank | 0.1573 | 0.3146 | нет |
| S2 | Vector+Rerank | 0.6547 | 0.6547 | нет |

S7 значимо лучше слабых стратегий (BM25, чистый Vector, мультиквери+реранк), но **статистически неотличим от других реранк-стратегий** (S2, S5).

### Сводный скор (формула §9 методики)

`Score = 0.35·MRR + 0.25·Recall@5 + 0.15·Precision@5 + 0.10·Hit@5 − 0.10·norm(t_поиска)`

| Стратегия | Score |
|---|---|
| **S2 Vector+Rerank** | **0.7093** |
| S7 Hybrid_RRF+Rerank | 0.7020 |
| S5 Hybrid+Rerank | 0.6940 |
| S6 Hybrid_RRF | 0.6912 |
| S4 Hybrid | 0.6886 |
| S1 Vector | 0.6827 |
| S3 BM25 | 0.6458 |
| S8 Hybrid+MultiQuery | 0.5849 |
| S9 Hybrid+MQ+Rerank | 0.5498 |

**Победитель: S2 Vector+Rerank.** Несмотря на чуть меньший MRR, чем у S7 (0.9508 vs 0.9545), S2 сохраняет Recall@5 = Hit@5 = 1.0 (реранкер в S7 вытесняет 2 правильных документа из топ-5), а формула §9 даёт Recall вес 0.25.

### Вариации top_k (S2 Vector+Rerank)

| top_k | R@5 | MRR | NDCG@5 | Hit@5 | t поиска, с |
|---|---|---|---|---|---|
| 3 | 0.9545 | 0.9280 | 0.9348 | 0.9545 | 18.7 |
| **5** | **1.0** | **0.9508** | **0.9635** | **1.0** | 18.8 |
| 10 | 1.0 | 0.9508 | 0.9635 | 1.0 | 18.6 |
| 15 | 1.0 | 0.9508 | 0.9635 | 1.0 | 18.8 |
| 20 | 1.0 | 0.9508 | 0.9635 | 1.0 | 18.8 |

Метрики выходят на плато при **top_k ≥ 5** и далее не меняются. При top_k=3 наблюдается лёгкая просадка (реранкеру не хватает кандидатов). Время поиска от top_k практически не зависит (~18.7 с) — стоимость определяется энкодингом запроса и реранком, а не размером пула. **Рекомендация: top_k = 10** (максимум качества + запас прочности на нестандартные запросы).

### Вариации RRF k

Неприменимо: выбранная стратегия S2 (Vector+Rerank) не использует RRF-слияние.

---

## Выводы

1. **Реранкер — главный фактор.** Все реранк-стратегии (S2/S5/S7) кучкуются вверху (MRR 0.93–0.95) и статистически неотличимы; не-реранк (S1/S4/S6) — ниже (MRR 0.87–0.90). Прирост от реранкинга ~6–9% MRR.

2. **Способ комбинирования кандидатов вторичен.** dense, hybrid и hybrid_RRF дают близкие результаты; различия незначимы. Поэтому выбрана самая простая реранк-стратегия — S2 (dense + cross-encoder, без BM25 и RRF).

3. **Мультиквери не оправдана.** S8/S9 — последние места по сводному скору из-за ~30-кратного времени поиска (~500 с) при отсутствии прироста качества. S9 — худшая стратегия по всем метрикам. **Фаза 2D (число подзапросов) не проводилась** как нецелесообразная.

4. **Модель: 4B достаточно.** 8B не даёт значимого преимущества над 4B на dim 2048 (p = 0.593). MRL-усечение 2560→2048 теряет <1% качества.

5. **Промпт: none лучше instruct** для Qwen на данном корпусе.

---

## Ограничения

- Малая статистическая мощность: 44 вопроса — на грани минимума методики (30). Многие p-value неинформативны из-за малого числа различающихся вопросов.
- Оценка на уровне документов (chunk→doc_id), а не отдельных чанков: исходная разметка chunk_id была привязана к другой сетке чанкования (512), несовместимой с продакшн-сеткой 1024. Документный уровень обеспечивает сопоставимость метрик между фазами.
- Бинарная релевантность (релевантен документ / нет); градации релевантности 0–3 из методики не задействованы.
- Overlap тестировался только на 10%.
- Ручная оценка качества ответов LLM (§3.2 методики) не проводилась — вне скоупа фазы 2.

---

## Пояснение терминов

**p-value** — вероятность получить наблюдаемую разницу между двумя вариантами (или большую) чисто за счёт случайности, если на самом деле разницы нет. Порог — 0.05: при p < 0.05 различие считается статистически значимым (реальным); при p > 0.05 различие не доказано (скорее всего шум на данной выборке). Важно: p > 0.05 означает «нет доказательств различия», а не «доказано равенство» — на малой выборке тесту может не хватать мощности уловить мелкие различия.

**Тест Уилкоксона (Wilcoxon signed-rank)** — непараметрический парный тест: сравнивает две стратегии по каждому из 44 вопросов попарно и проверяет, систематически ли одна лучше другой. Не требует нормального распределения метрик, поэтому подходит для небольших выборок.

**Поправка Holm-Bonferroni** — корректировка p-value при множественных сравнениях. Когда тестов много (9 стратегий → много пар), какой-то из них может «показать значимость» случайно. Поправка ужесточает порог пропорционально числу сравнений, отсеивая ложные срабатывания. Менее консервативна, чем простая поправка Бонферрони.

**Сводный скор (§9 методики)** — взвешенная свёртка метрик качества и стоимости в одно число для ранжирования стратегий:
`Score = 0.35·MRR + 0.25·Recall@5 + 0.15·Precision@5 + 0.10·Hit@5 − 0.10·norm(время поиска)`.
MRR имеет наибольший вес (важно, чтобы правильный ответ был первым), время поиска входит со штрафом. Веса заданы методикой.

**Метрики качества поиска:**
- **MRR** (Mean Reciprocal Rank) — насколько высоко в выдаче стоит первый релевантный результат (1.0 = всегда первым).
- **Recall@5** — доля релевантных документов, попавших в топ-5.
- **Precision@5** — доля релевантных среди топ-5.
- **Hit@5** — доля запросов, где хотя бы один релевантный документ есть в топ-5.
- **NDCG@5** — качество ранжирования топ-5 с учётом позиций (выше — лучше).

---

## Артефакты

| Файл | Содержание |
|---|---|
| `results/phase2a_chunksize.csv` | метрики по размеру чанка |
| `results/phase2b_models.csv` | 7 моделей + 4 MRL-прогона |
| `results/phase2c_strategies.csv` | 9 стратегий поиска |
| `results/phase2c_perq_mrr.csv` | per-question MRR (44×9) для статтестов |
| `results/phase2c_scored.csv` | сводный скор §9 |
| `results/phase2c_topk.csv` | вариации top_k (S2) |
| `results/report_phase2.md` | настоящий отчёт |

> Примечание: внутри ноутбука per-question MRR сохраняется в `.npz` (рабочий формат
> numpy для перезапуска статтестов). В репозиторий выложена эквивалентная
> CSV-версия `phase2c_perq_mrr.csv` (те же значения, 44 вопроса × 9 стратегий) —
> читаемая прямо в браузере и пригодная для повторных статистических сравнений.

'''
path = f"{BASE}/results/report_phase2.md"
with open(path, "w", encoding="utf-8") as f:
    f.write(report)
print("сохранено:", path, "|", len(report), "символов")


сохранено: /content/drive/MyDrive/rag_exp/results/report_phase2.md | 10663 символов


In [9]:
#@title Ячейка 11 - Упаковка результатов (проверить и упаковать для скачивания)
import os, zipfile

files = [
    "phase2a_chunksize.csv",
    "phase2b_models.csv",
    "phase2c_strategies.csv",
    "phase2c_perq_mrr.csv",
    "phase2c_scored.csv",
    "phase2c_topk.csv",
    "report_phase2.md",
]
res_dir = f"{BASE}/results"
print("=== что есть в results/ ===")
for f in files:
    p = os.path.join(res_dir, f)
    print(("  ЕСТЬ  " if os.path.exists(p) else "  НЕТ   ") + f)

zip_path = "/content/rag_phase2_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        p = os.path.join(res_dir, f)
        if os.path.exists(p):
            z.write(p, arcname=f"results/{f}")
print("\nархив:", zip_path)

from google.colab import files as colab_files
colab_files.download(zip_path)


=== что есть в results/ ===
  ЕСТЬ  phase2a_chunksize.csv
  ЕСТЬ  phase2b_models.csv
  ЕСТЬ  phase2c_strategies.csv
  ЕСТЬ  phase2c_perq_mrr.csv
  ЕСТЬ  phase2c_scored.csv
  ЕСТЬ  phase2c_topk.csv
  ЕСТЬ  report_phase2.md

архив: /content/rag_phase2_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>